In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [ ]:
# Tavily検索用のツールを定義
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://www.walkerplus.com/event_list/ar0313/sc309880d/',
   'title': '東京駅(東京都)周辺のイベント｜ウォーカープラス',
   'content': '# 東京駅(東京都)周辺のイベント. クロード・モネ《戸外の人物習作ー日傘を持つ右向きの女》1886年、油彩・カンヴァス、オルセー美術館蔵. 小林 清親《東京新大橋雨中図》明治9年(1876年) スミソニアン国立アジア美術館. 《平治物語絵巻 常盤巻》(部分)、 鎌倉時代13世紀、重要文化財、石橋財団アーティゾン美術館. CREATIVE MUSEUM TOKYO(クリエイティブ ミュージアム トウキョウ). ## エリアやカテゴリで絞り込む. ## よく使われる検索条件. ## テーマWalker. ## 季節特集. ## おでかけ特集. ## 東京都イベントランキング. ## 東京都おでかけスポットランキング. ## おすすめ記事. ## 閲覧履歴. ## おでかけトレンド. ## X(旧Twitter)で最新情報をチェック. ## 編集部イチオシ.',
   'score': 0.7711725,
   'raw_content': None},
  {'url': 'https://www.enjoytokyo.jp/event/list/area1306/',
   'title': "東京駅周辺・丸の内でおすすめのイベント - Let's ENJOY TOKYO",
   'content': '東京駅・丸の内の開催中または開催予定のイベントを紹介。おでかけに詳しい編集部がセレクトしました。気になるイベントは開催期間や開催場所、最寄駅、地図、料金など',
   'score': 0.741169,
   'raw_content': None},
  {'url': 'https://ekitan.com/event/station-2590',
   'title': '東京駅周辺のイベント - 駅探',
   'content': '東京駅周辺のイベント一覧。日付、料金等を指定しての絞込みも。お出かけに役立つ情報を案内。',
   'score': 0.70702106,
   'raw

In [5]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]
    
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [ ]:
# ツール呼び出し不要

tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- 東京都の面積は約2,194平方キロメートルです。
- 沖縄県の面積は約2,271平方キロメートルです。

したがって、広さに関しては沖縄県が東京都よりも大きいです。


In [7]:
# ツール呼び出しが必要な質問

tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近の東京駅でのイベントに関する情報は以下の通りです。

1. **JR東京駅 八重洲中央改札外イベントスペース**  
   - イベント内容: 新生アドベントカレンダーの展示  
   - 開催期間: 1ヶ月以上の申込み可能で、具体的な実施日は未設定です。  
   [詳細はこちら](https://koubunsya.com/ja/blog/jr%E6%9D%B1%E4%BA%AC%E9%A7%85-%E5%85%AB%E9%87%8D%E6%B4%B2%E4%B8%AD%E5%A4%AE%E6%94%B9%E6%9C%AD%E5%A4%96%E3%82%A4%E3%83%99%E3%83%B3%E3%83%88%E3%82%B9%E3%83%9A%E3%83%BC%E3%82%B9)

2. **グランスタ東京（駅構内）**  
   - イベント内容: 「POP UP GRANSTA GREEN」の開催  
   - 開催期間: 基本的に1ヶ月間の長期ポップアップショップ。  
   [詳細はこちら](https://prtimes.jp/main/html/rd/p/000000064.000152494.html)

3. **Instagramイベント**  
   - 内容: 1ヶ月遅れのバースデーイベントを開催  
   - 開催日: 6月25日(日) 18:00～22:00  
   [詳細はこちら](https://www.instagram.com/p/CtYh1r0x2pJ/)

4. **東京交通会館**  
   - 展示会やイベントの情報が掲載されています。詳細はサイトをご覧ください。  
   [詳細はこちら](https://www.kotsukaikan.co.jp/business/exhibition/)

5. **東京駅一番街**  
   - イベント内容: 「TVアニメ夏目友人帳 POP UP SHOP」  
   - 開催期間: 2026年4月10日～4月23日など、数つの限定ショップがオープンします。  
   [詳細はこちら](https://www.tokyoeki-1bangai.co.jp/event/)

この情報は、イベントの公式ウェブ

In [ ]:
# チャットボットへの組み込み
tools = define_tools()


sy = {"role": "system", "content": "あなたは親しい友達です。質問に対して、フレンドリーな口調で答えてください。"}

messages=[sy]

# 言語モデルへの質問を行う関数(システムプロンプトも受け取るように修正)
def ask_question(messages, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )
    return response


# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, messages):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    messages_past = messages.copy()
    messages_past.append(response.choices[0].message)
    messages_past.append({
        "role": "tool",
        "tool_call_id": tool.id,
        "content": function_response
    })

    # ツール呼び出し後の回答を取得
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages_past,
    )
    return response_after_tool_call


sy = {"role": "system","content": "あなたは親しい友達です。質問に対して、フレンドリーな口調で答えてください。"}
messages = [sy]

while True:
    question = input("メッセージを入力:")
    if question.strip() == "":
        break

    print(f"質問:{question}")
    messages.append({"role": "user", "content": question.strip()})

    response_message = process_response(messages, tools)

    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

    if len(messages) > 17: 
        messages.pop(1)  # systemプロンプトを残して古いメッセージから削除

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------
質問:こんにちは
こんにちは！元気にしてる？何か話したいことがあったら教えてね！
質問:寒暖差がすごいときのファッションどうしよう
寒暖差が激しいときは、重ね着が一番だよね！例えば、薄手のTシャツの上にカーディガンや軽めのジャケットを羽織って、暖かいときは脱ぐことができるようなスタイルが良いよ。あと、脱ぎやすいアウターや、軽くて持ち運びやすいストールなんかも便利だね！

靴は、通気性があって歩きやすいものを選ぶといいかも。それか、もし天気が不安定なら、レインコートもおすすめ！寒暖差を楽しみながら、オシャレを楽しんでね！
質問:大宮の花見シーズンっていつまで？
大宮の花見シーズンは、例年3月下旬から4月上旬がピークだよ。今年は、3月21日くらいから桜が咲き始めて、4月の初め頃が見頃だったみたい。天候によって変わることもあるけど、基本的にはその期間が狙い目かな。

大宮公園やその周辺では、桜を楽しむ人で賑わうから、ぜひお花見を楽しんでね！今から計画するのもいいかもしれないね！
質問:ひたちなか海浜公園について調べて
ひたちなか海浜公園は茨城県にある素敵な公園で、海と緑が融合した美しい場所だよ！春には色とりどりの花が楽しめるし、特にネモフィラの丘は有名で、青い花が一面に広がる光景が圧巻だよね。

公園内にはサイクリングや散歩を楽しめる道がたくさんあるし、バーベキューエリアも用意されているから、友達や家族と楽しいひとときを過ごすにはもってこいの場所だよ！興味があったら、公式サイトを見てみると、イベント情報や入園情報が詳しく載ってるからチェックしてみてね！ 

詳しくは [こちら](https://www.hitachikaihin.jp/) からどうぞ！

---ご利用ありがとうございました！---

---ご利用ありがとうございました！---
